# Data Verification — what's actually in the dataset

Confirms: how many players, which are **real match data** vs. **squad-only**
(haven't played), whether anything **fake** was injected, and overall provenance.
Run top-to-bottom.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/all_players_stats.parquet')
print('shape (rows, cols):', df.shape)
print('id dtypes -> player_id:', df['player_id'].dtype,
      '| team_id:', df['team_id'].dtype, '| game_id:', df['game_id'].dtype)
df.head(3)

## 1. Headline counts

In [ ]:
mins = pd.to_numeric(df['minutes'], errors='coerce').fillna(0)
print('total rows                 :', len(df))
print('unique player_id           :', df['player_id'].nunique())
print('unique games (game_id)     :', df['game_id'].nunique(dropna=True))
print('unique teams               :', df['team_name'].nunique())
print()
print('PLAYED (minutes > 0)       :', int((mins > 0).sum()))
print('SQUAD-ONLY (minutes == 0)  :', int((mins == 0).sum()), '<- added by scrape_squads.py')

## 2. Real match data vs. squad-only rows

A **real match appearance** always has a `game_id`. A **squad-only** player (hasn't
played yet) has `game_id = NaN` and `minutes = 0`, but a real ESPN identity.

In [ ]:
real_rows  = df['game_id'].notna().sum()
squad_rows = df['game_id'].isna().sum()
print(f'real match rows (has game_id)   : {real_rows}')
print(f'squad-only rows (game_id is NaN): {squad_rows}')
if squad_rows:
    print('\n-- sample squad-only players (real ESPN identities, 0 minutes) --')
    display(df[df['game_id'].isna()]
            [['player_id','player_name','team_name','position','role','minutes']].head(15))

## 3. Fake-data check

Real ESPN athlete IDs are 5–7 digit numbers (e.g. `159382`, `3098209`). The only
fabricated data ever generated was a throwaway demo using IDs `1000001–1000005`.
This checks those exact IDs are **absent**.

In [ ]:
DEMO_IDS = {'1000001','1000002','1000003','1000004','1000005'}
fake = df[df['player_id'].astype(str).isin(DEMO_IDS)]
print('fabricated demo players present:', len(fake), '(should be 0)')

pid = pd.to_numeric(df['player_id'], errors='coerce')
print('real ESPN id range:', int(pid.min()), '->', int(pid.max()))
print('any null player_id:', int(df['player_id'].isna().sum()))
print('\n=> If demo count is 0, every player is a real ESPN athlete.')

## 4. Squad completeness per team

`played` + `squad_only` = total players now selectable in the predictor for each
team. A full national squad is ~23–26.

In [ ]:
g = df.assign(_min=mins)
per_team = (g.groupby('team_name')
             .agg(total=('player_id','nunique'),
                  played=('_min', lambda s: (s>0).sum()),
                  squad_only=('_min', lambda s: (s==0).sum()))
             .sort_values('total', ascending=False))
per_team

## 5. Spot-check a player you can verify

Cross-check against ESPN / the broadcast. Try a squad-only name (e.g. an unused
sub) to confirm it's a real player with zero stats.

In [ ]:
name = 'Alphonso Davies'   # change to anyone
cols = ['player_name','team_name','game_id','minutes','touches',
        'expectedGoals','duelsWon']
cols = [c for c in cols if c in df.columns]
df[df['player_name'].str.contains(name, case=False, na=False)][cols]

## Verdict

- **Section 2** shows real match rows vs. real squad-only rows.
- **Section 3** = 0 fabricated demo players → all identities are real ESPN athletes
  pulled from `site.api.espn.com/.../teams/{id}/roster`.
- Squad-only players carry a real name/position but zero tournament stats, so the
  predictor treats them as untested (weak default ratings).